# Laplace–Beltrami PDE Solver on Curved Manifolds

This notebook solves the **heat**, **Schrödinger**, and **wave** equations on a
2D Riemannian manifold using the spectral `PDESolver`.  The Laplace–Beltrami
operator $\Delta_g$ is extracted from a `Metric` object and fed into the
solver as a `psiOp` pseudo-differential operator.

**Three coupled steps:**
1. Pick a metric from the catalogue (one cell to edit).
2. Choose a PDE type and run the solver.
3. Animate the solution either on the flat parameter domain or *on the 3-D
   embedding* of the manifold.

---

## Mathematical background

Given a Riemannian metric $g_{ij}$ in local coordinates $(x,y)$, the
Laplace–Beltrami operator acting on a scalar $u$ is

$$
\Delta_g u = \frac{1}{\sqrt{|g|}} \partial_i\!\left(\sqrt{|g|}\, g^{ij} \partial_j u\right).
$$

Its full symbol (principal + subprincipal) is

$$
\sigma(\Delta_g)(x,\xi) = \underbrace{g^{ij}\xi_i\xi_j}_{\text{principal}} +
  i\underbrace{\frac{1}{\sqrt{|g|}} \partial_i(\sqrt{|g|}\, g^{ij})\xi_j}_{\text{subprincipal}},
$$

and $\Delta_g$ acts in Fourier space as multiplication by $-\sigma(\Delta_g)$.

The three PDEs are:

| Equation | Formulation | Operator symbol |
|---|---|---|
| Heat | $\partial_t u = \Delta_g u$ | $-\sigma(\Delta_g)$ |
| Schrödinger | $i\partial_t u = -\Delta_g u$ | $-i\,\sigma(\Delta_g)$ |
| Wave | $\partial_{tt} u = \Delta_g u$ | $-\sigma(\Delta_g)$ (2nd order) |

## 0. Imports

In [ ]:
import numpy as np
import sympy as sp
from sympy import symbols, Matrix, I
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML

from riemannian import Metric, build_embedding, plot_embedding
from solver import PDESolver
from psiop import psiOp

import warnings
warnings.filterwarnings('ignore')

## 1. Metric catalogue

Pick one entry by setting `METRIC_NAME`.  Each entry is a dict with:

| key | meaning |
|---|---|
| `g` | 2×2 SymPy `Matrix` of metric components |
| `coords` | `(x, y)` coordinate symbols matching `g` |
| `domain` | `(Lx, Ly)` — physical **size** of the solver domain (centred at 0) |
| `shift` | `(x_shift, y_shift)` — offset added to solver coords before evaluating `g` |
| `ic_center` | centre of the Gaussian initial blob in **solver** coords |
| `ic_sigma` | width of the Gaussian |
| `Lt`, `Nt` | simulation time and number of time steps |
| `Nx`, `Ny` | spatial resolution |
| `description` | human-readable label |

> **Coordinate shift.**  The `PDESolver` always places its grid symmetrically
> around 0: $x\in[-L_x/2,L_x/2]$.  When the metric is ill-defined near 0
> (e.g. the Poincaré half-plane needs $y>0$), set `shift=(x0, y0)` so that
> the symbol is evaluated at the *shifted* coordinates $(x+x_0, y+y_0)$.

In [ ]:
# ── Coordinate symbols ───────────────────────────────────────────────────────
x, y = symbols('x y', real=True)

# ── Catalogue ────────────────────────────────────────────────────────────────
METRICS = {
    # ==========================================================================
    # Existing metrics (6)
    # ==========================================================================
    'flat': dict(
        g           = Matrix([[1, 0], [0, 1]]),
        coords      = (x, y),
        domain      = (4.0, 4.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 5.0,
        Nt          = 400,
        Nx          = 64, Ny = 64,
        description = 'Flat torus (K = 0)',
    ),
    'poincare': dict(
        g           = Matrix([[1/y**2, 0], [0, 1/y**2]]),
        coords      = (x, y),
        domain      = (4.0, 2.0),
        shift       = (0.0, 1.5),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.2,
        Lt          = 3.0,
        Nt          = 400,
        Nx          = 64, Ny = 64,
        description = 'Poincaré half-plane (K = -1)',
    ),
    'sphere': dict(
        g           = Matrix([[1, 0], [0, sp.sin(x)**2]]),
        coords      = (x, y),
        domain      = (2.54, 6.28),
        shift       = (1.57, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Unit sphere (K = +1)',
    ),
    'saddle': dict(
        g           = Matrix([[1, 0], [0, 1 + x**2 + y**2]]),
        coords      = (x, y),
        domain      = (4.0, 4.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 100,
        Nx          = 64, Ny = 64,
        description = 'Saddle surface (variable K < 0)',
    ),
    'cone': dict(
        g           = Matrix([[1, 0], [0, x**2]]),
        coords      = (x, y),
        domain      = (2.0, 6.28),
        shift       = (1.5, 0.0),
        ic_center   = (0.01, 0.01),
        ic_sigma    = 0.3,
        Lt          = 2.0,
        Nt          = 300,
        Nx          = 64, Ny = 64,
        description = 'Cone / polar  ds² = dr² + r²dθ²',
    ),

    # ==========================================================================
    # New metrics (14) – total 20
    # ==========================================================================

    # 1. Catenoid (minimal surface)
    #    ds² = cosh²(v) (du² + dv²)  with u ∈ [0,2π], v ∈ [-1.5,1.5]
    #    Gaussian curvature K = -1 / cosh⁴(v)  (negative, symmetric)
    'catenoid': dict(
        g           = Matrix([[sp.cosh(y)**2, 0], [0, sp.cosh(y)**2]]),
        coords      = (x, y),
        domain      = (6.283185, 3.0),      # u ∈ [-π, π], v ∈ [-1.5, 1.5]
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.4,
        Lt          = 3.0,
        Nt          = 200,
        Nx          = 64, Ny = 64,
        description = 'Catenoid (minimal surface, K < 0)',
    ),

    # 2. Pseudosphere (tractricoid) – constant negative curvature K = -1
    #    ds² = du² + sinh²(u) dv², u ∈ [0.5, 3.0], v ∈ [0, 2π]
    'pseudosphere': dict(
        g           = Matrix([[1, 0], [0, sp.sinh(x)**2]]),
        coords      = (x, y),
        domain      = (2.5, 6.283185),
        shift       = (1.0, 0.0),            # x_actual = x + 1.0 → u ∈ [0.5, 3.0]
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 200,
        Nx          = 64, Ny = 64,
        description = 'Pseudosphere (K = -1)',
    ),

    # 3. Elliptic paraboloid (z = x² + y²)
    #    ds² = (1+4x²)dx² + 8xy dxdy + (1+4y²)dy², K = 4/(1+4x²+4y²)²
    'paraboloid': dict(
        g           = Matrix([[1 + 4*x**2, 8*x*y], [8*x*y, 1 + 4*y**2]]),
        coords      = (x, y),
        domain      = (4.0, 4.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.4,
        Lt          = 3.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Elliptic paraboloid (positive curvature)',
    ),

    # 4. Schwarzschild 2D (spacelike slice, r > 2M, M=1)
    #    ds² = (1 - 2/r)^{-1} dr² + r² dφ², r ∈ [2.5, 5.0]
    'schwarzschild': dict(
        g           = Matrix([[1/(1 - 2/x), 0], [0, x**2]]),
        coords      = (x, y),
        domain      = (2.5, 6.283185),       # r ∈ [2.5, 5.0], φ ∈ [-π, π]
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 2.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Schwarzschild 2D (r > 2M)',
    ),

    # 5. Berger sphere (stretched sphere)
    #    ds² = dθ² + a² sin²θ dφ², a = 0.6 (oblate) -> variable K
    'berger_sphere': dict(
        g           = Matrix([[1, 0], [0, 0.36 * sp.sin(x)**2]]),
        coords      = (x, y),
        domain      = (2.54, 6.283185),
        shift       = (1.57, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Berger sphere (oblate, a=0.6)',
    ),

    # 6. Zoll metric (all geodesics closed)
    #    ds² = (1 + a cos θ)² dθ² + sin²θ dφ², a=0.3
    'zoll': dict(
        g           = Matrix([[(1 + 0.3*sp.cos(x))**2, 0], [0, sp.sin(x)**2]]),
        coords      = (x, y),
        domain      = (2.54, 6.283185),
        shift       = (1.57, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Zoll metric (a=0.3)',
    ),

    # 7. Parabolic coordinates metric
    #    ds² = (u²+v²)(du²+dv²) , u,v ∈ [-2,2]  (conformally flat)
    'parabolic': dict(
        g           = Matrix([[x**2 + y**2, 0], [0, x**2 + y**2]]),
        coords      = (x, y),
        domain      = (4.0, 4.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 2.0,
        Nt          = 200,
        Nx          = 64, Ny = 64,
        description = 'Parabolic coordinates (conformally flat, variable K)',
    ),

    # 8. Axisymmetric bump metric (conformally flat with Gaussian bump)
    #    ds² = e^{2U(r)} (dr² + r² dφ²), U(r) = 0.5 exp(-r²/2)
    'bump': dict(
        g           = Matrix([[sp.exp(2*0.5*sp.exp(-(x**2+y**2)/2)), 0],
                              [0, (x**2)*sp.exp(2*0.5*sp.exp(-(x**2+y**2)/2))]]),
        coords      = (x, y),
        domain      = (4.0, 6.283185),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 2.0,
        Nt          = 200,
        Nx          = 64, Ny = 64,
        description = 'Axisymmetric bump (K variable)',
    ),

    # 9. Anisotropic flat metric
    #    ds² = dx² + 2a dxdy + dy², a = 0.5
    'aniso_flat': dict(
        g           = Matrix([[1, 0.5], [0.5, 1]]),
        coords      = (x, y),
        domain      = (4.0, 4.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 5.0,
        Nt          = 400,
        Nx          = 64, Ny = 64,
        description = 'Anisotropic flat torus (K = 0)',
    ),

    # 10. Product metric S¹ × H¹ (2D)
    #     ds² = dθ² + (1/y²)(dx²+dy²) – needs 3 coordinates? No.
    #     Instead take a simpler hyperbolic cylinder: ds² = dθ² + dρ²/(ρ²) (ρ>0)
    #     but we already have Poincaré. We'll add a non‑diagonal hyperbolic metric.
    'hyperbolic_aniso': dict(
        g           = Matrix([[1/y**2, 0.2/y], [0.2/y, 1/y**2]]),
        coords      = (x, y),
        domain      = (4.0, 2.0),
        shift       = (0.0, 1.5),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.2,
        Lt          = 3.0,
        Nt          = 300,
        Nx          = 64, Ny = 64,
        description = 'Anisotropic Poincaré half-plane',
    ),

    # 11. Cylinder (flat product S¹ × ℝ)
    #     ds² = dx² + dy², but with periodic boundary in y.
    #     Here we simply use a flat metric on a rectangle with periodic BC.
    #     Already covered by 'flat', but we add a narrower domain to emphasize anisotropy.
    'cylinder': dict(
        g           = Matrix([[1, 0], [0, 1]]),
        coords      = (x, y),
        domain      = (6.0, 2.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.4,
        Lt          = 4.0,
        Nt          = 300,
        Nx          = 64, Ny = 64,
        description = 'Cylinder (flat, elongated x-direction)',
    ),

    # 12. Hyperbolic paraboloid (different from 'saddle')
    #     z = x² - y², induced metric: ds² = (1+4x²)dx² -8xy dxdy + (1+4y²)dy²
    'hyperbolic_paraboloid': dict(
        g           = Matrix([[1 + 4*x**2, -8*x*y], [-8*x*y, 1 + 4*y**2]]),
        coords      = (x, y),
        domain      = (4.0, 4.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Hyperbolic paraboloid (saddle with mixed signature?) – actually Riemannian, K<0',
    ),

    # 13. Schwarzschild–de Sitter 2D (Euclidean section)
    #     ds² = (1 - 2M/r - Λr²/3)^{-1} dr² + r² dφ², choose Λ small
    'schwarzschild_ds': dict(
        g           = Matrix([[1/(1 - 2/x - 0.1*x**2), 0], [0, x**2]]),
        coords      = (x, y),
        domain      = (3.0, 6.283185),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 2.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Schwarzschild–de Sitter 2D (M=1, Λ=0.3)',
    ),

    # 14. Rosenberg metric (an example of non‑constant positive curvature)
    #     ds² = (1 + a cos θ)² dθ² + sin²θ dφ² (already Zoll), so we add a different one:
    #     ds² = dθ² + (1 + ε cos θ) sin²θ dφ², ε=0.2
    'rosenberg': dict(
        g           = Matrix([[1, 0], [0, (1 + 0.2*sp.cos(x))*sp.sin(x)**2]]),
        coords      = (x, y),
        domain      = (2.54, 6.283185),
        shift       = (1.57, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Rosenberg metric (variable positive curvature)',
    ),
    # ==========================================================================
    # Additional interesting metrics (10) – total 30 with the previous set
    # ==========================================================================
    
    'klein': dict(
        g           = Matrix([[1, 0], [0, 1]]),
        coords      = (x, y),
        domain      = (4.0, 2.0),               # y‑periodic with a twist (non‑orientable)
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 5.0,
        Nt          = 400,
        Nx          = 64, Ny = 64,
        description = 'Flat Klein bottle (non‑orientable, K=0)',
    ),
    
    'clifford': dict(
        g           = Matrix([[1, 0], [0, 1]]),
        coords      = (x, y),
        domain      = (2*np.pi, 2*np.pi),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.4,
        Lt          = 5.0,
        Nt          = 400,
        Nx          = 64, Ny = 64,
        description = 'Clifford torus (square flat torus, minimal in S³)',
    ),
    
    'enneper': dict(
        g           = Matrix([[ (1 + x**2 + y**2)**2 , 0], [0, (1 + x**2 + y**2)**2 ]]),
        coords      = (x, y),
        domain      = (2.0, 2.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 200,
        Nx          = 64, Ny = 64,
        description = 'Enneper surface (conformally flat, K ≤ 0)',
    ),
    
    'bump_pos': dict(
        g           = Matrix([[1, 0], [0, 1]]) * sp.exp(2 * 0.5 * sp.cos(x)**2 * sp.cos(y)**2),
        coords      = (x, y),
        domain      = (4.0, 4.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 200,
        Nx          = 64, Ny = 64,
        description = 'Flat plane with a Gaussian bump (conformal factor, positive curvature)',
    ),
    
    'eggcarton': dict(
        g           = Matrix([[1, 0], [0, 1]]) * (1 + 0.5 * sp.sin(x)**2 * sp.sin(y)**2),
        coords      = (x, y),
        domain      = (2*np.pi, 2*np.pi),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 5.0,
        Nt          = 400,
        Nx          = 64, Ny = 64,
        description = 'Egg‑carton metric (periodic alternating curvature)',
    ),
    
    'warped_circle': dict(
        g           = Matrix([[1, 0], [0, (1 + 0.5*sp.cos(x))**2]]),
        coords      = (x, y),
        domain      = (2*np.pi, 2*np.pi),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 4.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Warped product S¹ ×_f S¹ (varying radius in y)',
    ),
    
    'minkowski': dict(
        g           = Matrix([[-1, 0], [0, 1]]),
        coords      = (x, y),
        domain      = (4.0, 4.0),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 3.0,
        Nt          = 200,
        Nx          = 64, Ny = 64,
        description = 'Minkowski plane (Lorentzian, K=0) – indefinite metric',
    ),
    
    'wavy_cylinder': dict(
        g           = Matrix([[1, 0], [0, (1 + 0.3*sp.cos(y))**2]]),
        coords      = (x, y),
        domain      = (2*np.pi, 2*np.pi),
        shift       = (0.0, 0.0),
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 4.0,
        Nt          = 300,
        Nx          = 64, Ny = 64,
        description = 'Cylinder with wavy radius (curvature varies along axis)',
    ),
    
    'extremal_rn': dict(
        g           = Matrix([[1/(1 - 2/x + 1/x**2), 0], [0, x**2]]),
        coords      = (x, y),
        domain      = (2.0, 6.283185),
        shift       = (1.0, 0.0),               # r_actual = x+1 → r from 2 to 6
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 2.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Extremal Reissner‑Nordström 2D (black hole, r > r_h)',
    ),
    
    'fischer_marsden': dict(
        g           = Matrix([[1, 0], [0, sp.sinh(x)**2]]),
        coords      = (x, y),
        domain      = (2.0, 6.283185),
        shift       = (1.0, 0.0),               # x_actual = x+1 → u from 2 to 6
        ic_center   = (0.0, 0.0),
        ic_sigma    = 0.3,
        Lt          = 2.0,
        Nt          = 150,
        Nx          = 64, Ny = 64,
        description = 'Fischer–Marsden metric (constant negative curvature)',
    ),
}
# ── Select one ────────────────────────────────────────────────────────────────
METRIC_NAME = 'berger_sphere'  # ← change this

cfg = METRICS[METRIC_NAME]
print(f"Selected metric: {cfg['description']}")

boundary_condition = 'dirichlet'  # or 'dirichlet' or 'neumann' or 'periodic'

## 2. Build the `Metric` and inspect the Laplace–Beltrami symbol

We apply the coordinate shift here so that the symbol is expressed in terms
of the *solver* coordinates `(x_s, y_s) = (x - x_shift, y - y_shift)`.  The
solver evaluates the symbol on its own grid, which is centred at zero.

In [ ]:
# ── Build Metric ──────────────────────────────────────────────────────────────
metric = Metric(cfg['g'], cfg['coords'])

print(f"Dimension    : {metric.dim}")
print(f"Coordinates  : {metric.coords}")
print(f"Curvature K  : {sp.simplify(metric.gauss_curvature())}")

# ── Laplace–Beltrami symbol ───────────────────────────────────────────────────
lb = metric.laplace_beltrami_symbol()
print("\nPrincipal symbol  σ₂ =", lb['principal'])
print("Subprincipal symbol σ₁ =", lb['subprincipal'])
print("Full symbol  σ = σ₂ + i σ₁ =", lb['full'])

In [ ]:
# ── Apply coordinate shift to the symbol ─────────────────────────────────────
x_coord, y_coord = cfg['coords']
x_shift, y_shift = cfg['shift']

def shift_symbol(sym_expr):
    """Substitute (x → x + x_shift, y → y + y_shift) in a symbol expression."""
    subs = {}
    if x_shift != 0.0:
        subs[x_coord] = x_coord + x_shift
    if y_shift != 0.0:
        subs[y_coord] = y_coord + y_shift
    return sym_expr.subs(subs) if subs else sym_expr

full_sym_shifted = shift_symbol(lb['full'])
print("Shifted full symbol:")
sp.pprint(full_sym_shifted)

## 3. PDE equation builders

Each builder returns a SymPy `Eq` that the `PDESolver` can parse.  The
relationship between the LBO symbol $\sigma$ and the `psiOp` argument is:

$$
\Delta_g u \;\leftrightarrow\; \mathtt{psiOp}(-\sigma,\, u)
$$

because the operator is defined with the **negative** symbol (the
positive-definite elliptic operator $-\Delta_g$ has positive principal
symbol $+g^{ij}\xi_i\xi_j$).

In [ ]:
def make_heat_eq(u_func, symbol_expr):
    """
    Heat equation:  ∂_t u = Δ_g u

    Parameters
    ----------
    u_func : applied SymPy Function, e.g. u(t, x, y)
    symbol_expr : full Laplace–Beltrami symbol (shifted if necessary)
    """
    t_var = u_func.args[0]
    return sp.Eq(sp.diff(u_func, t_var), psiOp(-symbol_expr, u_func))


def make_schrodinger_eq(u_func, symbol_expr):
    """
    Schrödinger equation:  i ∂_t u = -Δ_g u   ⟺   ∂_t u = i Δ_g u

    Parameters
    ----------
    u_func : applied SymPy Function, e.g. u(t, x, y)
    symbol_expr : full Laplace–Beltrami symbol (shifted if necessary)
    """
    t_var = u_func.args[0]
    return sp.Eq(sp.diff(u_func, t_var), psiOp(-I * symbol_expr, u_func))


def make_wave_eq(u_func, symbol_expr):
    """
    Wave equation:  ∂_tt u = Δ_g u

    Parameters
    ----------
    u_func : applied SymPy Function, e.g. u(t, x, y)
    symbol_expr : full Laplace–Beltrami symbol (shifted if necessary)
    """
    t_var = u_func.args[0]
    return sp.Eq(sp.diff(u_func, t_var, t_var), psiOp(-symbol_expr, u_func))


print("PDE builders ready.")

## 4. Shared simulation helpers

In [ ]:
# ── Symbolic function ─────────────────────────────────────────────────────────
t_sym = sp.Symbol('t', real=True, positive=True)
x_sym, y_sym = sp.symbols('x y', real=True)
u_sym = sp.Function('u')(t_sym, x_sym, y_sym)

# ── Grid parameters (from catalogue) ─────────────────────────────────────────
Lx, Ly     = cfg['domain']
Nx, Ny     = cfg['Nx'], cfg['Ny']
Lt, Nt     = cfg['Lt'], cfg['Nt']
x0_ic, y0_ic = cfg['ic_center']
sigma_ic   = cfg['ic_sigma']

# ── Initial conditions ────────────────────────────────────────────────────────
def gaussian_ic(X, Y):
    """Normalised 2-D Gaussian blob centred at (x0_ic, y0_ic)."""
    return np.exp(-((X - x0_ic)**2 + (Y - y0_ic)**2) / (2 * sigma_ic**2))

def zero_velocity(X, Y):
    """Zero initial velocity for the wave equation."""
    return np.zeros_like(X)

print(f"Domain : x ∈ [{-Lx/2:.2f}, {Lx/2:.2f}],  y ∈ [{-Ly/2:.2f}, {Ly/2:.2f}]")
print(f"Grid   : {Nx} × {Ny},  dt = {Lt/Nt:.4f}")

## 5. Solve – Heat equation

In [ ]:
eq_heat   = make_heat_eq(u_sym, full_sym_shifted)
slv_heat  = PDESolver(eq_heat)
slv_heat.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=gaussian_ic,
    boundary_condition=boundary_condition,
    plot=False,
)
print("Solving heat equation …")
frames_heat = slv_heat.solve()
print(f"Done.  Stored {len(frames_heat)} frames.")

In [ ]:
ani_heat = slv_heat.animate(component='abs', mode='surface', overlay='front')
HTML(ani_heat.to_jshtml())

## 6. Solve – Schrödinger equation

In [ ]:
eq_schro  = make_schrodinger_eq(u_sym, full_sym_shifted)
slv_schro = PDESolver(eq_schro, time_scheme='default')
slv_schro.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=gaussian_ic,
    boundary_condition=boundary_condition,
    plot=False,
)
print("Solving Schrödinger equation …")
frames_schro = slv_schro.solve()
print(f"Done.  Stored {len(frames_schro)} frames.")

In [ ]:
ani_schro = slv_schro.animate(component='abs', mode='surface', overlay='front')
HTML(ani_schro.to_jshtml())

## 7. Solve – Wave equation

In [ ]:
eq_wave  = make_wave_eq(u_sym, full_sym_shifted)
slv_wave = PDESolver(eq_wave)
slv_wave.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=gaussian_ic,
    initial_velocity=zero_velocity,
    boundary_condition=boundary_condition,
    plot=False,
)
print("Solving wave equation …")
frames_wave = slv_wave.solve()
print(f"Done.  Stored {len(frames_wave)} frames.")

In [ ]:
ani_wave = slv_wave.animate(component='abs', mode='surface', overlay='front')
HTML(ani_wave.to_jshtml())

## 8. 3-D embedding of the manifold

`build_embedding` constructs an approximate isometric embedding
$R : \Omega \to \mathbb{R}^3$ of the parameter domain into 3-D space by
marching row-by-row while maintaining a Darboux frame.  The result is a
numpy array of shape `(nu, nv, 3)`.

> **Note.** The parameter ranges passed to `build_embedding` must match the
> *actual* physical coordinates (i.e. the *shifted* grid).

In [ ]:
# Actual physical coordinate ranges (solver grid + shift)
u_range = (-Lx/2 + x_shift, Lx/2 + x_shift)
v_range = (-Ly/2 + y_shift, Ly/2 + y_shift)

print(f"Embedding parameter ranges:")
print(f"  u (= x_actual) ∈ [{u_range[0]:.3f}, {u_range[1]:.3f}]")
print(f"  v (= y_actual) ∈ [{v_range[0]:.3f}, {v_range[1]:.3f}]")

# Use the same resolution as the solver for a 1-to-1 pixel correspondence
R_embed, u_vals, v_vals = build_embedding(metric, u_range, v_range, Nx, Ny)
print(f"\nEmbedding shape: {R_embed.shape}   (nu={Nx}, nv={Ny}, xyz=3)")

In [ ]:
# ── Quick static view of the embedding ───────────────────────────────────────
plot_embedding(R_embed, title=f'Embedding — {cfg["description"]}', colormap='plasma', dark=True);

## 9. Animate the PDE solution *on the embedding*

The key idea is straightforward: the solver stores one `(Nx, Ny)` solution
array per frame, and the embedding `R` has shape `(Nx, Ny, 3)`.  At each
animation frame we colour the surface `(X, Y, Z)` with the corresponding
solution value — the geometry is fixed, only the colormap changes.

This works because the solver and the embedding share the **same parameter
grid** (both use `Nx × Ny` points over the same physical domain).

In [ ]:
def animate_on_embedding(
    R, frames, title='',
    component='abs', cmap='plasma',
    dark=True, interval=80, n_anim_frames=60,
):
    """
    Animate a scalar PDE solution on a 3-D embedding.

    Parameters
    ----------
    R : ndarray, shape (nu, nv, 3)
        The static 3-D embedding.  Its (nu, nv) grid must match that of
        every frame in `frames`.
    frames : list of ndarray, each shape (nu, nv)
        Solution snapshots (may be complex).  Produced by ``solver.solve()``.
    title : str
        Figure title.
    component : {'real', 'imag', 'abs', 'angle'}
        Which component of the (possibly complex) solution to display.
    cmap : str
        Matplotlib colormap name.
    dark : bool
        Dark (True) or light (False) background.
    interval : int
        Milliseconds between frames.
    n_anim_frames : int
        Number of animation frames (uniformly sampled from `frames`).

    Returns
    -------
    ani : matplotlib.animation.FuncAnimation
    """
    def _get_component(u):
        ops = {'real': np.real, 'imag': np.imag,
               'abs': np.abs,  'angle': np.angle}
        if component not in ops:
            raise ValueError(f"component must be one of {list(ops)}.")
        return ops[component](u)

    bg = '#111111' if dark else 'white'
    tc = 'white'   if dark else 'black'

    X, Y, Z = R[:,:,0], R[:,:,1], R[:,:,2]

    # Sample frames uniformly
    idx = np.linspace(0, len(frames) - 1, n_anim_frames, dtype=int)
    sampled = [_get_component(frames[i]) for i in idx]

    # Global colour scale for perceptual consistency across frames
    vmin = min(f.min() for f in sampled)
    vmax = max(f.max() for f in sampled)
    if vmax - vmin < 1e-12:
        vmin, vmax = 0.0, 1.0

    cm_func = plt.colormaps[cmap]

    fig = plt.figure(figsize=(8, 6), facecolor=bg)
    ax  = fig.add_subplot(111, projection='3d', facecolor=bg)
    ax.set_axis_off()

    def _norm(arr):
        return (arr - vmin) / (vmax - vmin)

    # Draw first frame
    surf = [ax.plot_surface(
        X, Y, Z,
        facecolors=cm_func(_norm(sampled[0])),
        linewidth=0, antialiased=True, shade=True,
    )]
    time_text = ax.text2D(
        0.02, 0.95, 't = 0.00',
        transform=ax.transAxes, color=tc, fontsize=10,
    )
    ax.set_title(title, color=tc, fontsize=11, pad=8)

    # Stable camera angle
    ax.view_init(elev=25, azim=-60)

    # Add a fixed colorbar (mappable with global scale)
    mappable = plt.cm.ScalarMappable(cmap=cmap)
    mappable.set_clim(vmin, vmax)
    cb = plt.colorbar(mappable, ax=ax, shrink=0.55, aspect=18, pad=0.02)
    cb.ax.yaxis.set_tick_params(color=tc, labelcolor=tc)
    cb.set_label(component, color=tc)

    def _update(frame_number):
        surf[0].remove()
        surf[0] = ax.plot_surface(
            X, Y, Z,
            facecolors=cm_func(_norm(sampled[frame_number])),
            linewidth=0, antialiased=True, shade=True,
        )
        t_val = (idx[frame_number] / (len(frames) - 1)) * slv_wave.Lt
        time_text.set_text(f't = {t_val:.2f}')
        return (surf[0],)

    ani = FuncAnimation(
        fig, _update,
        frames=n_anim_frames,
        interval=interval, blit=False,
    )
    plt.tight_layout()
    return ani

print("animate_on_embedding() defined.")

### 9a. Heat equation on the embedding

In [ ]:
ani_heat_3d = animate_on_embedding(
    R_embed, frames_heat,
    title=f'Heat equation — {cfg["description"]}',
    component='abs', cmap='inferno',
)
HTML(ani_heat_3d.to_jshtml())

### 9b. Schrödinger equation on the embedding

In [ ]:
ani_schro_3d = animate_on_embedding(
    R_embed, frames_schro,
    title=f'Schrödinger equation (|ψ|) — {cfg["description"]}',
    component='abs', cmap='viridis',
)
HTML(ani_schro_3d.to_jshtml())

### 9c. Wave equation on the embedding

In [ ]:
ani_wave_3d = animate_on_embedding(
    R_embed, frames_wave,
    title=f'Wave equation — {cfg["description"]}',
    component='abs', cmap='coolwarm',
)
HTML(ani_wave_3d.to_jshtml())

## 10. Side-by-side snapshot: parameter domain vs. embedding

Static comparison at the last stored frame, useful for publication figures.

In [ ]:
def snapshot_comparison(
    R, frames, solver, frame_idx=-1,
    component='abs', cmap='plasma', dark=True,
    pde_label='PDE',
):
    """
    Two-panel figure: flat domain (imshow) on the left,
    3-D embedding (surface) on the right.

    Parameters
    ----------
    R : ndarray (nu, nv, 3) — embedding
    frames : list of (nu, nv) arrays — solver frames
    solver : PDESolver — used only for grid info
    frame_idx : int — which frame to display (-1 = last)
    """
    def _get(u):
        ops = {'real': np.real, 'imag': np.imag,
               'abs': np.abs,  'angle': np.angle}
        return ops[component](u)

    bg = '#111111' if dark else 'white'
    tc = 'white'   if dark else 'black'

    scalar = _get(frames[frame_idx])
    X3, Y3, Z3 = R[:,:,0], R[:,:,1], R[:,:,2]

    vmin, vmax = scalar.min(), scalar.max()
    if vmax - vmin < 1e-12:
        vmin, vmax = 0.0, 1.0

    fig = plt.figure(figsize=(13, 5), facecolor=bg)

    # ── Left: flat imshow ────────────────────────────────────────────────────
    ax_flat = fig.add_subplot(1, 2, 1, facecolor=bg)
    extent  = [solver.x_grid[0], solver.x_grid[-1],
               solver.y_grid[0], solver.y_grid[-1]]
    im = ax_flat.imshow(
        scalar.T, origin='lower', extent=extent,
        cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto',
    )
    ax_flat.set_title(f'{pde_label} — parameter domain', color=tc)
    ax_flat.set_xlabel('x (solver)', color=tc)
    ax_flat.set_ylabel('y (solver)', color=tc)
    ax_flat.tick_params(colors=tc)
    for sp_ in ax_flat.spines.values():
        sp_.set_color('#444' if dark else 'gray')
    plt.colorbar(im, ax=ax_flat, shrink=0.85)

    # ── Right: 3-D embedding ─────────────────────────────────────────────────
    ax_3d = fig.add_subplot(1, 2, 2, projection='3d', facecolor=bg)
    s_norm = (scalar - vmin) / (vmax - vmin)
    ax_3d.plot_surface(
        X3, Y3, Z3,
        facecolors=plt.colormaps[cmap](s_norm),
        linewidth=0, antialiased=True, shade=True,
    )
    ax_3d.set_title(f'{pde_label} — embedding', color=tc)
    ax_3d.set_axis_off()
    ax_3d.view_init(elev=25, azim=-55)

    plt.suptitle(cfg['description'], color=tc, fontsize=12)
    plt.tight_layout()
    plt.show()


# ── Heat ──────────────────────────────────────────────────────────────────────
snapshot_comparison(R_embed, frames_heat, slv_heat,
                    component='abs', cmap='inferno',
                    pde_label='Heat equation')

# ── Schrödinger ───────────────────────────────────────────────────────────────
snapshot_comparison(R_embed, frames_schro, slv_schro,
                    component='abs', cmap='viridis',
                    pde_label='Schrödinger equation')

# ── Wave ──────────────────────────────────────────────────────────────────────
snapshot_comparison(R_embed, frames_wave, slv_wave,
                    component='real', cmap='coolwarm',
                    pde_label='Wave equation')